# 04 · Ejecutar Python en Hadoop con Streaming

Una celda Python normal se ejecuta en Jupyter. Aquí enviamos un trabajo MapReduce a **YARN**, que ejecuta los scripts en NodeManager. Un solo portátil no equivale a varias máquinas físicas.

El trabajo cuenta palabras. Crea una salida nueva en cada ejecución; Hadoop exige que la carpeta de salida aún no exista.

In [ ]:
from pathlib import Path
from uuid import uuid4
import subprocess
from clientes import ALUMNO, BASE, hdfs
local = Path("datos") / ALUMNO / "streaming"
local.mkdir(parents=True, exist_ok=True)
(local / "palabras.txt").write_text("hadoop hive hadoop\ntrino hive\n", encoding="utf-8")
(local / "mapper.py").write_text("import sys\nfor line in sys.stdin:\n    for word in line.split():\n        print(f'{word}\t1')\n", encoding="utf-8")
(local / "reducer.py").write_text("import sys\nfrom itertools import groupby\nrows = (line.rstrip().split('\t') for line in sys.stdin)\nfor word, values in groupby(rows, key=lambda row: row[0]):\n    print(f'{word}\t{sum(int(row[1]) for row in values)}')\n", encoding="utf-8")
hdfs("-mkdir", "-p", f"{BASE}/streaming/entrada")
hdfs("-put", "-f", str(local / "palabras.txt"), f"{BASE}/streaming/entrada/")

In [ ]:
salida = f"{BASE}/streaming/salida_{uuid4().hex[:8]}"
comando = ["hadoop", "jar", "/opt/bd/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar",
    "-D", "mapreduce.job.name=BDA-Python-" + ALUMNO,
    "-D", "mapreduce.job.reduces=1",
    "-files", f"{local / 'mapper.py'},{local / 'reducer.py'}",
    "-mapper", "python3 mapper.py", "-reducer", "python3 reducer.py",
    "-input", f"{BASE}/streaming/entrada", "-output", salida]
resultado = subprocess.run(comando, text=True, capture_output=True, timeout=600)
print(resultado.stdout)
print(resultado.stderr)
resultado.check_returncode()

In [ ]:
texto = hdfs("-cat", salida + "/part-*")
conteos = dict(linea.split("\t") for linea in texto.strip().splitlines())
assert conteos == {"hadoop": "2", "hive": "2", "trino": "1"}
print("MapReduce Python validado; salida:", salida)

Abre http://localhost:8088 para ver la aplicación **BDA-Python-…**, su estado y sus tareas. Esta vez sí interviene YARN.

**Ejercicio:** modifica el mapper para convertir palabras a minúsculas. Añade palabras con mayúsculas al archivo, vuelve a subirlo y compara el resultado.

Las rutas anteriores permanecen en HDFS; no uses `down -v` para parar el laboratorio.